In [ ]:
import random

In [ ]:
import numpy as np
import pandas as pd
from pycaret.regression import get_config
from pycaret.regression import set_config
from pycaret.regression import (
    setup,
    compare_models,
    create_model,
    tune_model,
    ensemble_model,
    models,
    blend_models,
    stack_models,
    plot_model,
    evaluate_model,
    interpret_model,
    automl,
    predict_model,
    save_model,
    load_model
)
from pycaret.utils import version

In [ ]:
id_cols = ['productID']
target_cols = ['WS']
predictors = [
    'productStock',
    'price',
    'averageRating_bestRating',
    'averageRating_worstRating',
    'bestRating',
    'worstRating',
    'prodAlcoholVolume_text',
    'prodAlcoholPercent_percent',
    'JS', 'WW', 'D', 'BH',
    'W&S', 'WE', 'RP', 'JD',
    'SJ', 'V', 'CG', 'TP'
]

In [ ]:
print(f"pycaret version = {version()}")
print("1. Loading Dataset")
df = pd.read_csv("wine_spectator.csv")
indices = list(df.index)
n_samples = len(indices)
train_samples = int(0.8 * n_samples)
test_samples = n_samples - train_samples
print(f"train_samples = {train_samples}")
print(f"test_samples = {test_samples}")
random.shuffle(indices)
pick_k_random_indices = random.sample(indices, train_samples)
not_picked_indices = set(indices).difference(pick_k_random_indices)
df_train = df.loc[pick_k_random_indices, :]
df_train_id = df_train[id_cols]
df_train = df_train.drop(id_cols, axis=1)
df_train = df_train.dropna(subset=target_cols, axis=0)

In [ ]:
df_test = df.loc[not_picked_indices, :]
df_test_id = df_test[id_cols]
df_test_truth = df_test[target_cols]
df_test = df_test.drop(id_cols, axis=1)
df_test = df_test.drop(target_cols, axis=1)

In [ ]:
print(list(df.describe()))

In [ ]:
print("2. Initialize Setup")
reg1 = setup(
    data=df_train,
    target="WS",
    experiment_name='ws_from_meta',
    imputation_type='iterative'
)

In [ ]:
print("3. Compare Baseline")

In [ ]:
best_model = compare_models(fold=5)

In [ ]:
print("4. Create Model")
lightgbm = create_model('lightgbm')
lgbms = [create_model('lightgbm', learning_rate=i) for i in np.arange(0.1, 1, 0.1)]
print(len(lgbms))

In [ ]:
print("5. Tune Hyperparameters")
tuned_lightgbm = tune_model(lightgbm, n_iter=50, optimize='MAE')

In [ ]:
print(tuned_lightgbm)

In [ ]:
print("6. Ensemble Model")

In [ ]:
dt = create_model('dt')

In [ ]:
bagged_dt = ensemble_model(dt, n_estimators=50)

In [ ]:
boosted_dt = ensemble_model(dt, method='Boosting')

In [ ]:
print("7. Blend Models")

In [ ]:
top_five = compare_models(n_select=5, fold=5, include=list(models().index))

In [ ]:
blender = blend_models(estimator_list=top_five)

In [ ]:
print("8. Stack Models")

In [ ]:
stacker = stack_models(estimator_list=top_five)

In [ ]:
print("9. Analyze Model")

In [ ]:
plot_model(dt)

In [ ]:
plot_model(dt, plot='error')

In [ ]:
plot_model(dt, plot='feature')

In [ ]:
evaluate_model(dt)

In [ ]:
print("10. Interpret Model")

In [ ]:
interpret_model(lightgbm)

In [ ]:
interpret_model(lightgbm, plot='correlation')

In [ ]:
interpret_model(lightgbm, plot='reason', observation=12)

In [ ]:
print("11. AutoML()")

In [ ]:
best = automl(optimize='MAE')
print(best)

In [ ]:
print("12. Predict Model")

In [ ]:
pred_holdouts = predict_model(lightgbm)
pred_holdouts.head()

In [ ]:
predict_new = predict_model(best, data=df_test)
predict_new.head()

In [ ]:
save_model(best, model_name='best-model')

In [ ]:
loaded_bestmodel = load_model('best-model')
print(loaded_bestmodel)

In [ ]:
set_config(display='diagram')
print(loaded_bestmodel[0])

In [ ]:
X_train = get_config('X_train')
X_train.head()